In [1]:
#all imports needed for EDA
from huggingface_hub import hf_hub_download
import duckdb
import pandas as pd
import numpy as np
import tarfile
from typing import Dict, Any, List, Optional

In [2]:
repo_name = "AmazonScience/WikiDT"

split_files = {
    "train" : "WikiDT-dataset/samples/train.json",
    "val" : "WikiDT-dataset/samples/val.json",
    "test" : "WikiDT-dataset/samples/test.json"
}


In [3]:
#Download the split files from the Hugging Face Hub
combined_data = {}
for split_name, file_path in split_files.items():
    try:
        path = hf_hub_download(repo_id=repo_name, filename=file_path, repo_type="dataset")
        combined_data[split_name] = path
    except Exception:
        print(f"Failed to download {file_path} from {repo_name}. Trying next split...")
        continue

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


In [4]:
#Download the OCR tar file from the Hugging Face Hub
ocr_tar_path = hf_hub_download(
    repo_id=repo_name,
    filename="WikiDT-dataset/ocr.tar",
    repo_type="dataset",
)

# Load files from train, test, and validate and combine it to one

In [5]:
con = duckdb.connect()

stage_info = []

for split_name, file_path in combined_data.items():
    print(f"Loading {split_name} from {file_path}...")
    stage_split_info = con.execute(f"""
        SELECT
            sample_id,
            image,
            question,
            json_extract_string(answer, '$[0][0]') AS answer,
            ocr_retrieval_file_textract,
            all_ocr_files_textract,
            '{split_name}' AS original_split
        FROM read_json(
            '{file_path}',
            columns={{
                'sample_id': 'VARCHAR',
                'image': 'VARCHAR',
                'question': 'VARCHAR',
                'answer': 'JSON',
                'ocr_retrieval_file_textract': 'VARCHAR',
                'all_ocr_files_textract': 'VARCHAR[]'
            }}
        )
    """).df()
    stage_info.append(stage_split_info)

df_all = pd.concat(stage_info, ignore_index=True)
total_combined_rows = len(df_all)
print(f"Total pooled QA pairs loaded: {total_combined_rows:,}")


Loading train from /root/.cache/huggingface/hub/datasets--AmazonScience--WikiDT/snapshots/db803a4c2031b7170063fed7bc0c445cb8aec9f6/WikiDT-dataset/samples/train.json...
Loading val from /root/.cache/huggingface/hub/datasets--AmazonScience--WikiDT/snapshots/db803a4c2031b7170063fed7bc0c445cb8aec9f6/WikiDT-dataset/samples/val.json...
Loading test from /root/.cache/huggingface/hub/datasets--AmazonScience--WikiDT/snapshots/db803a4c2031b7170063fed7bc0c445cb8aec9f6/WikiDT-dataset/samples/test.json...
Total pooled QA pairs loaded: 70,652


In [6]:
# Load the OCR tar file and index its members
print("Indexing members in ocr.tar...")
with tarfile.open(ocr_tar_path, "r") as tar:
    tar_members = set(tar.getnames())

available_textract_files = set()
for m in tar_members:
    clean = m.lstrip("./")
    if clean.startswith("WikiDT-dataset/"):
        clean = clean.replace("WikiDT-dataset/", "")
    if clean.startswith("ocr/textract/"):
        available_textract_files.add(clean)

print(f"Total available Textract OCR files in archive: {len(available_textract_files):,}")

Indexing members in ocr.tar...
Total available Textract OCR files in archive: 52,654


In [7]:
#Define a function to map each row to its corresponding OCR file, checking both primary and fallback options
def audit_row_coverage(row):
    primary = str(row["ocr_retrieval_file_textract"]).lstrip("./")
    if primary in available_textract_files:
        return "primary_match", primary

    fallbacks = row["all_ocr_files_textract"]
    if isinstance(fallbacks, (list, np.ndarray)):
        for fb in fallbacks:
            clean_fb = str(fb).lstrip("./")
            if clean_fb in available_textract_files:
                return "fallback_match", clean_fb

    return "missing", None

audit_results = df_all.apply(audit_row_coverage, axis=1)
df_all["mapping_status"] = [res[0] for res in audit_results]
df_all["resolved_ocr_path"] = [res[1] for res in audit_results]

In [8]:
# Apply the audit function to the DataFrame and summarize the results
statuses = [res[0] for res in audit_results]
primary_count = statuses.count("primary_match")
fallback_count = statuses.count("fallback_match")
missing_count = statuses.count("missing")
total_rows = len(statuses)

print(f"Primary Target Matches:   {primary_count:>7,} ({primary_count / total_rows * 100:>6.2f}%)")
print(f"Fallback Segment Matches: {fallback_count:>7,} ({fallback_count / total_rows * 100:>6.2f}%)")
print(f"Missing (Dropped):        {missing_count:>7,} ({missing_count / total_rows * 100:>6.2f}%)")
print(f"Total Usable OCR Rows:    {primary_count + fallback_count:>7,} ({(primary_count + fallback_count) / total_rows * 100:>6.2f}%)")

Primary Target Matches:    69,316 ( 98.11%)
Fallback Segment Matches:   1,209 (  1.71%)
Missing (Dropped):            127 (  0.18%)
Total Usable OCR Rows:     70,525 ( 99.82%)


In [9]:
# Filter the DataFrame to retain only rows with valid OCR mappings
df_filtered = df_all[df_all["mapping_status"] != "missing"].copy().reset_index(drop=True)
total_retained_rows = len(df_filtered)

print(f"Total Initial Pooled Rows:     {total_combined_rows:>8,}")
print(f"Retained (Valid OCR Target):   {total_retained_rows:>8,} ({total_retained_rows / total_combined_rows * 100:>6.2f}%)")
print(f"Dropped (Missing OCR):         {total_combined_rows - total_retained_rows:>8,} ({(total_combined_rows - total_retained_rows) / total_combined_rows * 100:>6.2f}%)")
print(f"Unique Retained Documents:     {df_filtered['image'].nunique():>8,}")

Total Initial Pooled Rows:       70,652
Retained (Valid OCR Target):     70,525 ( 99.82%)
Dropped (Missing OCR):              127 (  0.18%)
Unique Retained Documents:       14,148


In [10]:
def filter_by_column_rules(
    df: pd.DataFrame,
    rules: Dict[str, Dict[str, Any]],
    how: str = "all"
) -> pd.DataFrame:
    """
    Filters a DataFrame using per-column configurations.

    rules structure:
    {
        "column_name": {
            "allow_null": bool,          # True to keep NaN/None, False to drop them
            "exclude_values": list       # String tokens to drop (e.g. ["", "error"])
        }
    }
    """
    validity_matrix = pd.DataFrame(index=df.index)

    for col, config in rules.items():
        if col not in df.columns:
            continue

        allow_null = config.get("allow_null", False)
        exclude_values = {
            str(val).strip().lower() 
            for val in config.get("exclude_values", [])
        }

        # 1. Null handling
        is_null = df[col].isnull()
        if allow_null:
            null_condition = pd.Series(True, index=df.index)
        else:
            null_condition = ~is_null

        # 2. Token exclusion handling (only check non-null entries)
        cleaned_str = df[col].fillna("__NA_PLACEHOLDER__").astype(str).str.strip().str.lower()
        token_condition = ~cleaned_str.isin(exclude_values)

        validity_matrix[col] = null_condition & token_condition

    final_mask = validity_matrix.all(axis=1) if how == "all" else validity_matrix.any(axis=1)
    return df[final_mask]

In [11]:
rules = {
    "question": {
        "allow_null": False,
        "exclude_values": ["", "none", "nan", "null"]
    },
    "answer": {
        "allow_null": True,         # Keeps genuine nulls (unanswerable QA pairs)
        "exclude_values": ["", "none", "nan", "null"]      # Drops only empty string artifacts
    }
}

df_clean = filter_by_column_rules(df_filtered, rules=rules)

In [12]:
#Standarize answer with null value to be no answer
df_clean['answer'] = df_clean['answer'].fillna('no answer')

/tmp/ipykernel_17532/2586995965.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['answer'] = df_clean['answer'].fillna('no answer')


In [13]:
df_clean.columns

Index(['sample_id', 'image', 'question', 'answer',
       'ocr_retrieval_file_textract', 'all_ocr_files_textract',
       'original_split', 'mapping_status', 'resolved_ocr_path'],
      dtype='object')

In [14]:
df_clean.head(5)

,sample_id,image,question,answer,ocr_retrieval_file_textract,all_ocr_files_textract,original_split,mapping_status,resolved_ocr_path
0,0,204_232_page.png,was chile below or above the gold medal count ...,no answer,ocr/textract/204_232_page_seg_1.json,"[ocr/textract/204_232_page_seg_0.json, ocr/tex...",train,primary_match,ocr/textract/204_232_page_seg_1.json
1,1,203_700_page.png,in 2012 who was expected to light the tree?,no answer,ocr/textract/203_700_page_seg_0.json,[ocr/textract/203_700_page_seg_0.json],train,primary_match,ocr/textract/203_700_page_seg_0.json
2,2,204_833_page.png,which player scored the largest number of poin...,no answer,ocr/textract/204_833_page_seg_3.json,"[ocr/textract/204_833_page_seg_0.json, ocr/tex...",train,primary_match,ocr/textract/204_833_page_seg_3.json
3,3,203_490_page.png,number of wins in the season.,no answer,ocr/textract/203_490_page_seg_0.json,[ocr/textract/203_490_page_seg_0.json],train,primary_match,ocr/textract/203_490_page_seg_0.json
4,4,203_76_page.png,"which driver in the top 8, drives a mclaran-me...",no answer,ocr/textract/203_765_page_seg_0.json,"[ocr/textract/203_76_page_seg_0.json, ocr/text...",train,fallback_match,ocr/textract/203_76_page_seg_0.json


In [15]:
print(f"Total Rows Before Cleaning: {len(df_filtered)}")
print(f"Total Rows After Cleaning: {len(df_clean):,} ({len(df_clean) / len(df_filtered) * 100:>6.2f}% reduction)")

Total Rows Before Cleaning: 70525
Total Rows After Cleaning: 70,469 ( 99.92% reduction)


In [16]:
#Drop any columns that are not needed for the final dataset
df_clean = df_clean[["sample_id", "image", "question", "answer", "resolved_ocr_path"]]

In [17]:
def check_no_placeholders(
    df: pd.DataFrame,
    columns: Optional[List[str]] = None,
    invalid_tokens: Optional[List[str]] = None,
    raise_error: bool = False
) -> pd.DataFrame:
    """
    Performs a final validation across columns to verify no missing values,
    whitespace strings, or invalid string tokens remain.
    
    Returns a summary report DataFrame. If raise_error=True, raises an
    AssertionError when any violations are detected.
    """
    if invalid_tokens is None:
        invalid_tokens = ["", "none", "nan", "null"]
    
    # Store tokens in lowercase for case-insensitive matching
    banned_tokens = {str(t).strip().lower() for t in invalid_tokens}
    target_cols = columns if columns is not None else df.columns.tolist()

    report_records = []

    for col in target_cols:
        series = df[col]

        # 1. Check for genuine null values (NaN, pd.NA, None)
        true_nulls = series.isnull().sum()

        # 2. Check for whitespace, empty strings, and string tokens
        # Cast only non-nulls to string to avoid turning NaN into "nan" twice
        non_null_strs = series.dropna().astype(str).str.strip().str.lower()
        token_matches = non_null_strs.isin(banned_tokens).sum()

        total_violations = true_nulls + token_matches

        report_records.append({
            "Column": col,
            "True Nulls": true_nulls,
            "Placeholder Strings": token_matches,
            "Total Violations": total_violations,
            "Status": "PASS" if total_violations == 0 else "FAIL"
        })

    report_df = pd.DataFrame(report_records)

    total_issues = report_df["Total Violations"].sum()
    if total_issues == 0:
        print("Final Check: PASS — No nulls or banned placeholder tokens found.")
    else:
        print(f"Final Check: FAIL — Found {total_issues:,} total violations across columns.")

    if raise_error and total_issues > 0:
        failed_cols = report_df[report_df["Status"] == "FAIL"]["Column"].tolist()
        raise AssertionError(f"Validation failed for columns: {failed_cols}")

    return report_df

In [18]:
# Perform the final validation check on the cleaned DataFrame

report = check_no_placeholders(df_clean)
report

Final Check: PASS — No nulls or banned placeholder tokens found.


,Column,True Nulls,Placeholder Strings,Total Violations,Status
0,sample_id,0,0,0,PASS
1,image,0,0,0,PASS
2,question,0,0,0,PASS
3,answer,0,0,0,PASS
4,resolved_ocr_path,0,0,0,PASS


In [19]:
# Show the shape of the cleaned DataFrame
df_clean.shape

(70469, 5)